# <font color="#418FDE" size="6.5" uppercase>**Manuell vorverarbeiten**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Bereinigen fehlende, doppelte, unplausible und uneinheitliche Datenwerte. 
- Transformieren numerische, kategoriale, Text- und Datumsmerkmale manuell. 
- Entwerfen eine überprüfbare Vorverarbeitungsfunktion mit Kontrolltabellen. 


## **1. Daten bereinigen**

### **1.1. Fehlwerte sinnvoll ersetzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_01.jpg?v=1784784865" width="250">



>* Fehlwerte haben unterschiedliche Ursachen
>* Ersetzung braucht begründete Annahmen

>* Numerische Fehlwerte passend mit Median oder Mittelwert ersetzen
>* Kategorien und Kontext bei Ersetzungen beachten

>* Ersetzungen dokumentieren und später prüfen
>* Fehlende Werte als Information markieren



In [ ]:
#@title Python-Code - Fehlwerte sinnvoll ersetzen

# Wir ersetzen Fehlwerte mit begründeten einfachen Regeln.
# Median und Kategorie unbekannt bleiben nachvollziehbar.
# Eine Kontrolltabelle zeigt die ersetzten Werte.

import pandas as pd

# Dieser kleine Datensatz enthält typische fehlende Kundendaten.
data = pd.DataFrame(
    {
        "customer_id": [101, 102, 103, 104, 105, 106],
        "age": [34, None, 29, 41, None, 38],
        "monthly_spend_eur": [52.5, 80.0, None, 47.0, 120.0, None],
        "contact_method": ["E-Mail", None, "Telefon", "E-Mail", None, "Chat"],
    }
)

# Wir prüfen eine einfache Annahme zur Datensatzgröße.
if data.shape != (6, 4):
    raise ValueError("Der Beispieldatensatz hat eine unerwartete Form.")

# Vor der Ersetzung zählen wir die fehlenden Werte je Spalte.
missing_before = data.isna().sum()

# Wir arbeiten auf einer Kopie, damit Rohdaten erhalten bleiben.
cleaned_data = data.copy()

# Numerische Fehlwerte ersetzen wir hier robust mit dem Median.
age_median = cleaned_data["age"].median()
spend_median = cleaned_data["monthly_spend_eur"].median()

cleaned_data["age"] = cleaned_data["age"].fillna(age_median)
cleaned_data["monthly_spend_eur"] = cleaned_data["monthly_spend_eur"].fillna(spend_median)

# Kategoriale Fehlwerte bekommen eine eigene aussagekräftige Kategorie.
cleaned_data["contact_method"] = cleaned_data["contact_method"].fillna("unbekannt")

# Nach der Ersetzung prüfen wir, ob noch Fehlwerte übrig sind.
missing_after = cleaned_data.isna().sum()

# Die Kontrolltabelle dokumentiert Anzahl und Regel je Spalte.
control_table = pd.DataFrame(
    {
        "fehlend_vorher": missing_before,
        "fehlend_nachher": missing_after,
        "regel": ["keine", "Median", "Median", "unbekannt"],
    }
)

# Für eine kompakte Ausgabe zeigen wir nur betroffene Spalten.
affected_columns = ["age", "monthly_spend_eur", "contact_method"]
control_table = control_table.loc[affected_columns]

print("Bereinigte Beispieldaten:")
print(cleaned_data.head(5).to_string(index=False))
print("Kontrolltabelle:")
print(control_table.to_string())



### **1.2. Duplikate und Datentypen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_02.jpg?v=1784784867" width="250">



>* Duplikate können Analysen deutlich verzerren
>* Vor dem Löschen fachlich eindeutig prüfen

>* Datentypen passend und eindeutig wählen
>* Uneinheitliche Formate vor Analysen vereinheitlichen

>* Probleme zuerst prüfen und Muster erkennen
>* Bereinigung dokumentieren, Bedeutung der Daten bewahren



In [ ]:
#@title Python-Code - Duplikate und Datentypen

# Wir bereinigen Duplikate und falsche Datentypen.
# Pandas zeigt typische Probleme in Tabellendaten.
# Am Ende vergleichen wir Rohdaten und Bereinigung.

import pandas as pd

# Diese kleine Tabelle enthält absichtlich typische Datenprobleme.
raw_data = pd.DataFrame(
    {
        "order_id": ["A001", "A001", "A002", "A003", "A004"],
        "customer_id": ["K01", "K01", "K02", "K03", "K04"],
        "order_date": ["01.02.2024", "01.02.2024", "2024-02-03", "31.02.2024", ""],
        "amount_eur": ["19,99 €", "19,99 €", "5.50", "unbekannt", "-3,00 €"],
        "postal_code": ["01234", "01234", "80331", "10115", "04109"],
    }
)

# Eine einfache Prüfung schützt vor unerwartet leeren Beispieldaten.
if len(raw_data) == 0:
    raise ValueError("Die Beispieldaten dürfen nicht leer sein.")

# Exakte Duplikate erkennen wir über fachlich eindeutige Spalten.
duplicate_columns = ["order_id", "customer_id", "order_date", "amount_eur"]
duplicate_count = raw_data.duplicated(subset=duplicate_columns).sum()

# Für Beträge entfernen wir Symbole und vereinheitlichen Dezimalzeichen.
clean_data = raw_data.drop_duplicates(subset=duplicate_columns).copy()
clean_amount = clean_data["amount_eur"].str.replace("€", "", regex=False)
clean_amount = clean_amount.str.replace(",", ".", regex=False).str.strip()

# Fehlerhafte Zahlen werden zu fehlenden Werten statt zu falschen Zahlen.
clean_data["amount_eur"] = pd.to_numeric(clean_amount, errors="coerce")
clean_data.loc[clean_data["amount_eur"] < 0, "amount_eur"] = pd.NA

# Datumswerte werden eindeutig als Tag-Monat-Jahr interpretiert.
clean_data["order_date"] = pd.to_datetime(
    clean_data["order_date"], dayfirst=True, errors="coerce"
)

# Postleitzahlen bleiben Text, damit führende Nullen erhalten bleiben.
clean_data["postal_code"] = clean_data["postal_code"].astype("string")

# Eine Kontrolltabelle macht die Bereinigung nachvollziehbar.
control_table = pd.DataFrame(
    {
        "Kennzahl": ["Zeilen vorher", "Duplikate", "Zeilen nachher", "fehlende Beträge", "fehlende Daten"],
        "Wert": [len(raw_data), int(duplicate_count), len(clean_data), int(clean_data["amount_eur"].isna().sum()), int(clean_data["order_date"].isna().sum())],
    }
)

print("Kontrolltabelle zur Bereinigung:")
print(control_table.to_string(index=False))
print("Datentypen nach der Bereinigung:")
print(clean_data[["order_date", "amount_eur", "postal_code"]].dtypes.to_string())



### **1.3. Ausreißer markieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_03.jpg?v=1784784869" width="250">



>* Ausreißer können Fehler oder wichtige Hinweise sein
>* Markieren macht spätere Entscheidungen nachvollziehbar

>* Statistik zeigt Auffälligkeiten, Kontext erklärt sie
>* Markierungen bewahren Werte für spätere Prüfung

>* Ausreißerregeln nachvollziehbar dokumentieren
>* Auffällige Werte prüfen, nicht automatisch löschen



In [ ]:
#@title Python-Code - Ausreißer markieren

# Dieses Beispiel markiert Ausreißer ohne Werte zu löschen.
# Die IQR-Regel findet statistisch auffällige Einkommen.
# Eine Kontrolltabelle macht die Bereinigung nachvollziehbar.

import pandas as pd

# Wir erstellen kleine Beispieldaten mit Euro-Beträgen.
data = pd.DataFrame(
    {
        "person_id": [1, 2, 3, 4, 5, 6, 7, 8],
        "income_eur": [2100, 2300, 2500, 2600, 2700, 2900, 3100, 25000],
    }
)

# Diese Prüfung schützt vor einer leeren Spalte.
if data["income_eur"].notna().sum() < 4:
    raise ValueError("Für die IQR-Regel werden mindestens vier Werte benötigt.")

# Quartile beschreiben den typischen mittleren Wertebereich.
q1 = data["income_eur"].quantile(0.25)
q3 = data["income_eur"].quantile(0.75)
iqr = q3 - q1

# Die Grenzen markieren ungewöhnlich niedrige oder hohe Werte.
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

# Wir markieren Ausreißer, statt sie sofort zu entfernen.
cleaned = data.copy()
cleaned["is_outlier"] = (
    (cleaned["income_eur"] < lower_limit)
    | (cleaned["income_eur"] > upper_limit)
)

# Eine Begründung macht die Markierung später überprüfbar.
cleaned["check_note"] = "unauffällig"
cleaned.loc[cleaned["is_outlier"], "check_note"] = "außerhalb der IQR-Grenzen"

# Die Kontrolltabelle zeigt nur die wichtigsten Spalten.
control_table = cleaned.loc[
    cleaned["is_outlier"], ["person_id", "income_eur", "check_note"]
]

print("IQR-Grenzen für Einkommen in Euro:")
print(f"untere Grenze: {lower_limit:.0f}, obere Grenze: {upper_limit:.0f}")
print("Markierte Ausreißer:")
print(control_table.to_string(index=False))



## **2. Merkmale manuell transformieren**

### **2.1. Einheiten und Texte**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_01.jpg?v=1784784878" width="250">



>* Einheiten vereinheitlichen und korrekt umrechnen
>* Unklare Werte fachlich plausibel prüfen

>* Textvarianten sorgfältig vereinheitlichen
>* Bedeutung und wichtige Unterschiede bewahren

>* Regeln klar dokumentieren und wiederholbar anwenden
>* Uneindeutige Werte fachlich prüfen und markieren



In [ ]:
#@title Python-Code - Einheiten und Texte

# Wir vereinheitlichen Einheiten und kurze Textwerte.
# Manuelle Regeln machen Rohdaten vergleichbar und prüfbar.
# Die Kontrolltabelle zeigt bereinigte Werte und Warnungen.

import pandas as pd

# Kleine Rohdaten zeigen typische uneinheitliche Eingaben.
raw_data = pd.DataFrame(
    {
        "product": ["  Handy", "Smartphone", "Mobiltelefon", "Laptop", "handy"],
        "weight_raw": ["70 kg", "154 lb", "70 Kilogramm", "2,5 kg", "unbekannt"],
    }
)

# Diese Zuordnung fasst fachlich gleiche Textwerte zusammen.
category_map = {
    "handy": "Smartphone",
    "smartphone": "Smartphone",
    "mobiltelefon": "Smartphone",
    "laptop": "Laptop",
}

# Diese Funktion wandelt Gewichte in Kilogramm um.
def parse_weight_to_kg(value):
    text = str(value).strip().lower().replace(",", ".")
    number_text = "".join(ch for ch in text if ch.isdigit() or ch == ".")
    if number_text == "":
        return None
    number = float(number_text)
    if "lb" in text:
        return round(number * 0.453592, 1)
    return round(number, 1)

# Textwerte werden bereinigt und über die Regelkarte vereinheitlicht.
cleaned = raw_data.copy()
cleaned["product_key"] = cleaned["product"].str.strip().str.lower()
cleaned["product_clean"] = cleaned["product_key"].map(category_map)

# Unklare Texte bleiben sichtbar, statt still verloren zu gehen.
cleaned["product_clean"] = cleaned["product_clean"].fillna("Prüfen")
cleaned["weight_kg"] = cleaned["weight_raw"].apply(parse_weight_to_kg)
cleaned["status"] = "ok"

# Fehlende Umrechnungen werden für die Kontrolle markiert.
missing_weight = cleaned["weight_kg"].isna()
cleaned.loc[missing_weight, "status"] = "Gewicht prüfen"

# Die Kontrolltabelle zeigt Rohwert, Zielwert und Prüfhinweis.
control_table = cleaned[
    ["product", "product_clean", "weight_raw", "weight_kg", "status"]
]

print("Kontrolltabelle nach manueller Transformation:")
print(control_table.to_string(index=False))
print("Ziel-Einheit: Kilogramm; Textziel: einheitliche Produktkategorien.")



### **2.2. Merkmale skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_02.jpg?v=1784784876" width="250">



>* Große Zahlenwerte können Modelle verzerren
>* Skalierung macht Merkmale fair vergleichbar

>* Standardisierung und Min-Max passend auswählen
>* Schiefe Verteilungen robust behandeln

>* Nur echte numerische Merkmale skalieren
>* Skalierung dokumentieren und konsistent anwenden



In [ ]:
#@title Python-Code - Merkmale skalieren

# Dieses Beispiel skaliert numerische Merkmale manuell.
# Standardisierung und Min-Max-Skalierung werden verglichen.
# Die Ausgabe zeigt vergleichbare Wertebereiche.

import pandas as pd

# Kleine Beispieldaten zeigen unterschiedliche Größenordnungen.
data = pd.DataFrame(
    {
        "income_eur": [32000, 45000, 52000, 61000, 120000],
        "living_area_m2": [45, 62, 80, 95, 140],
        "satisfaction": [2, 3, 4, 4, 5],
    }
)

# Wir prüfen, ob alle Spalten gleich lang sind.
if data.shape != (5, 3):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Standardisierung nutzt Mittelwert und Standardabweichung.
means = data.mean()
stds = data.std(ddof=0)
standardized = (data - means) / stds

# Min-Max-Skalierung bringt Werte in den Bereich null bis eins.
mins = data.min()
ranges = data.max() - data.min()
min_max_scaled = (data - mins) / ranges

# Eine Kontrolltabelle macht die Transformation nachvollziehbar.
check_table = pd.DataFrame(
    {
        "original_income": data["income_eur"],
        "standardized_income": standardized["income_eur"].round(2),
        "minmax_income": min_max_scaled["income_eur"].round(2),
    }
)

print("Originale Wertebereiche:")
print(data.agg(["min", "max"]).round(2))
print("Skalierte Einkommen als Kontrolltabelle:")
print(check_table)
print("Nach Standardisierung: Mittelwert ungefähr 0, Streuung ungefähr 1.")



### **2.3. Binning und Log**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_03.jpg?v=1784784879" width="250">



>* Binning gruppiert Zahlen zu verständlichen Klassen
>* Klassengrenzen fachlich sinnvoll und interpretierbar wählen

>* Gruppenanzahl zwischen Genauigkeit und Übersicht abwägen
>* Klassengrenzen prüfen und fachlich begründen

>* Log komprimiert große, rechtsschiefe Werte.
>* Interpretation, Nullen und Dokumentation prüfen.



In [ ]:
#@title Python-Code - Binning und Log

# Dieses Beispiel zeigt Binning und Log-Transformation.
# Wir vergleichen Rohwerte mit abgeleiteten Merkmalen.
# Die Ausgabe macht Verteilungen leichter interpretierbar.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Kleine Beispieldaten bleiben vollständig im Arbeitsspeicher.
income_eur = np.array([1200, 1500, 1800, 2200, 2600, 3100, 3900, 5200, 9000])

# Die Tabelle enthält Rohwerte und zwei manuelle Transformationen.
data = pd.DataFrame({"income_eur": income_eur})
data["income_group"] = pd.cut(
    data["income_eur"], bins=[0, 2000, 4000, 10000], labels=["niedrig", "mittel", "hoch"]
)

# Log1p funktioniert auch bei Nullwerten und bleibt hier sicher.
data["log_income"] = np.log1p(data["income_eur"])

# Eine einfache Prüfung verhindert unbemerkte fehlende Gruppen.
if data["income_group"].isna().any():
    raise ValueError("Mindestens ein Einkommen passt in keine Gruppe.")

# Die Kontrolltabelle zeigt, ob die Gruppen sinnvoll besetzt sind.
control_table = data["income_group"].value_counts(sort=False).reset_index()
control_table.columns = ["Gruppe", "Anzahl"]

print("Kontrolltabelle nach dem Binning:")
print(control_table.to_string(index=False))
print("Log-Werte komprimieren besonders hohe Einkommen sichtbar.")

# Ein Streudiagramm vergleicht Originalskala und Log-Skala direkt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(data["income_eur"], data["log_income"], s=70)
ax.set_title("Einkommen: Originalwert gegen Log-Transformation")
ax.set_xlabel("Einkommen in Euro")
ax.set_ylabel("log1p(Einkommen)")
ax.grid(True, alpha=0.3)
plt.show()



## **3. Merkmale überprüfbar erzeugen**

### **3.1. Kategorien nachvollziehbar kodieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_01.jpg?v=1784784872" width="250">



>* Kodierung muss nachvollziehbar und stabil bleiben
>* Kontrollen verbinden Rohdaten mit Modellmerkmalen

>* Kodierung passend zur Kategorienordnung wählen
>* Varianten und seltene Kategorien kontrolliert behandeln

>* Kodierung als festen Prozessschritt dokumentieren
>* Kontrolltabellen sichern Wartbarkeit und Qualität



In [ ]:
#@title Python-Code - Kategorien nachvollziehbar kodieren

# Dieses Beispiel kodiert Kategorien nachvollziehbar.
# Kontrolltabellen zeigen Rohwerte und erzeugte Merkmale.
# Unbekannte Werte werden sichtbar und stabil behandelt.

import pandas as pd

# Kleine Rohdaten enthalten Schreibvarianten und fehlende Werte.
raw_data = pd.DataFrame(
    {
        "payment_raw": ["Kreditkarte", "KK", "PayPal", "Rechnung", None, "Bar"],
        "amount_eur": [120, 80, 45, 200, 60, 30],
    }
)

# Diese Zuordnung ist die dokumentierte fachliche Entscheidung.
payment_mapping = {
    "Kreditkarte": "Kreditkarte",
    "KK": "Kreditkarte",
    "PayPal": "PayPal",
    "Rechnung": "Rechnung",
}

# Die Funktion erzeugt Merkmale und eine Kontrolltabelle.
def preprocess_payments(input_data, mapping):
    data = input_data.copy()
    data["payment_clean"] = data["payment_raw"].map(mapping)
    data["payment_clean"] = data["payment_clean"].fillna("Unbekannt")

    encoded = pd.get_dummies(data["payment_clean"], prefix="pay", dtype=int)
    expected_columns = ["pay_Kreditkarte", "pay_PayPal", "pay_Rechnung", "pay_Unbekannt"]
    encoded = encoded.reindex(columns=expected_columns, fill_value=0)

    processed = pd.concat([data[["amount_eur"]], encoded], axis=1)
    control = data.groupby(["payment_raw", "payment_clean"], dropna=False).size()
    control = control.reset_index(name="count")
    return processed, control

# Eine einfache Prüfung schützt vor unerwarteten Spalten.
processed_data, control_table = preprocess_payments(raw_data, payment_mapping)
expected_width = 5
if processed_data.shape[1] != expected_width:
    raise ValueError("Die Anzahl der erzeugten Spalten stimmt nicht.")

# Die Ausgabe bleibt kurz und zeigt die Prüfbarkeit.
print("Kontrolltabelle: Rohwert -> bereinigte Kategorie -> Anzahl")
print(control_table.to_string(index=False))
print("Erzeugte Modellspalten:", list(processed_data.columns))
print("Unbekannte Fälle:", int(processed_data["pay_Unbekannt"].sum()))



### **3.2. Interaktionen erstellen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_02.jpg?v=1784784874" width="250">



>* Interaktionen zeigen abhängige Merkmalswirkungen
>* Nur fachlich begründete Kombinationen erzeugen

>* Interaktionen klar definieren und dokumentieren
>* Neue Spalten auf Auffälligkeiten prüfen

>* Interaktionen transparent und reproduzierbar kontrollieren
>* Datenlecks und Verzerrungen kritisch prüfen



In [ ]:
#@title Python-Code - Interaktionen erstellen

# Wir erzeugen überprüfbare Interaktionen aus Kundendaten.
# Kontrolltabellen zeigen fehlende Werte und Verteilungen.
# Das Ergebnis ist eine nachvollziehbare Vorverarbeitung.

import pandas as pd

# Kleine Beispieldaten machen jede neue Spalte nachvollziehbar.
raw_data = pd.DataFrame(
    {
        "customer_group": ["Stamm", "Neu", "Stamm", "Neu", "Stamm"],
        "visits": [8, 1, 5, 2, 10],
        "revenue_eur": [240.0, 35.0, 160.0, 50.0, 500.0],
        "discount_eur": [20.0, 0.0, 15.0, 5.0, 60.0],
    }
)

# Diese Funktion erzeugt Interaktionen reproduzierbar.
def preprocess_customers(input_data):
    data = input_data.copy()
    data["revenue_per_visit"] = data["revenue_eur"] / data["visits"]
    data["loyal_discount_eur"] = 0.0
    loyal_mask = data["customer_group"] == "Stamm"
    data.loc[loyal_mask, "loyal_discount_eur"] = data.loc[loyal_mask, "discount_eur"]
    data["value_interaction"] = data["revenue_per_visit"] * data["visits"]
    return data

# Die Vorverarbeitung verändert die Rohdaten nicht direkt.
processed_data = preprocess_customers(raw_data)

# Eine einfache Prüfung schützt vor unerwarteten Zeilenverlusten.
if len(processed_data) != len(raw_data):
    raise ValueError("Die Zeilenanzahl hat sich unerwartet verändert.")

# Kontrolltabelle eins prüft fehlende Werte der neuen Merkmale.
new_columns = ["revenue_per_visit", "loyal_discount_eur", "value_interaction"]
missing_check = processed_data[new_columns].isna().sum().to_frame("fehlend")

# Kontrolltabelle zwei zeigt Wertebereiche der Interaktionen.
range_check = processed_data[new_columns].agg(["min", "max"]).round(2)

# Eine kleine Stichprobe verbindet Rohwerte und neue Merkmale.
preview_columns = ["customer_group", "visits", "revenue_per_visit"]
preview_columns = preview_columns + ["loyal_discount_eur", "value_interaction"]
preview = processed_data[preview_columns].head(3).round(2)

print("Kontrolle fehlender Werte:")
print(missing_check)
print("Wertebereiche der neuen Merkmale:")
print(range_check)
print("Stichprobe der erzeugten Interaktionen:")
print(preview.to_string(index=False))



### **3.3. Vorverarbeitung praktisch prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_03.jpg?v=1784784870" width="250">



>* Merkmale auf Plausibilität und Änderungen prüfen
>* Kodierungen nachvollziehbar und erklärbar dokumentieren

>* Kontrolltabellen bündeln wichtige Prüfinformationen kompakt
>* Sie begleiten Vorverarbeitung und zeigen Fehler früh

>* Prüfungen wiederholbar und vergleichbar gestalten
>* Abweichungen erkennen und Ergebnisse vertrauenswürdig machen



In [ ]:
#@title Python-Code - Vorverarbeitung praktisch prüfen

# Dieses Beispiel prüft manuelle Vorverarbeitung praktisch.
# Kontrolltabellen machen Änderungen und Grenzfälle sichtbar.
# Am Ende entsteht ein kurzes Prüfprotokoll.

import pandas as pd

# Kleine Rohdaten enthalten typische Probleme aus der Praxis.
raw_data = pd.DataFrame(
    {
        "customer_id": [101, 102, 102, 103, 104, 105],
        "birth_year": [1988, 2005, 2005, 1890, None, 1975],
        "group": ["Student", "student", "VIP", "vip", "", "Standard"],
        "purchase_eur": [49.9, 0.0, 0.0, -5.0, 120.0, None],
    }
)

# Die Funktion bereinigt Daten und sammelt Kontrollwerte.
def preprocess_customers(data):
    cleaned = data.copy()
    checks = []

    checks.append(["Startzeilen", len(cleaned), "Rohdaten"])
    duplicate_count = cleaned.duplicated(subset=["customer_id"]).sum()
    checks.append(["Doppelte IDs", int(duplicate_count), "vor Entfernung"])

    cleaned = cleaned.drop_duplicates(subset=["customer_id"], keep="first")
    cleaned["group_clean"] = cleaned["group"].str.strip().str.lower()
    cleaned["group_clean"] = cleaned["group_clean"].replace("", "unknown")

    cleaned["age"] = 2026 - cleaned["birth_year"]
    invalid_age = cleaned["age"].isna() | (cleaned["age"] < 0) | (cleaned["age"] > 100)
    cleaned["age_flag"] = invalid_age.map({True: "prüfen", False: "ok"})

    invalid_purchase = cleaned["purchase_eur"].isna() | (cleaned["purchase_eur"] < 0)
    cleaned["purchase_flag"] = invalid_purchase.map({True: "prüfen", False: "ok"})

    checks.append(["Endzeilen", len(cleaned), "nach Dubletten"])
    checks.append(["Alter prüfen", int(invalid_age.sum()), "fehlend/unplausibel"])
    checks.append(["Kauf prüfen", int(invalid_purchase.sum()), "fehlend/negativ"])

    check_table = pd.DataFrame(checks, columns=["Prüfung", "Anzahl", "Hinweis"])
    return cleaned, check_table

# Die Kontrolltabelle zeigt, ob die Verarbeitung plausibel wirkt.
processed_data, check_table = preprocess_customers(raw_data)
print("Kontrolltabelle:")
print(check_table.to_string(index=False))

# Eine kleine Stichprobe verbindet Originalwerte mit neuen Merkmalen.
preview_columns = ["customer_id", "group_clean", "age", "age_flag", "purchase_flag"]
print("Beispielzeilen:")
print(processed_data[preview_columns].head(3).to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**Manuell vorverarbeiten**</font>


In this lecture, you learned to:
- Bereinigen fehlende, doppelte, unplausible und uneinheitliche Datenwerte. 
- Transformieren numerische, kategoriale, Text- und Datumsmerkmale manuell. 
- Entwerfen eine überprüfbare Vorverarbeitungsfunktion mit Kontrolltabellen. 

In the next Module (Module 5), we will go over 'Bilder und Signale'